<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/appendix/A_wandb_skills.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Appendix A：wandb/skills — AI エージェント向けスキル

`wandb/skills` は、Claude Code・Cursor・Codex などの AI コーディングエージェントが
W&B / Weave を正確に扱えるよう設計された公式スキルパッケージです。

インストールすると、エージェントは W&B SDK と Weave SDK の正しい使い方・注意点・
ヘルパー関数を「知識」として持ち、より精度の高いコード生成・データ分析を行えます。

## 学習内容
1. wandb/skills の概要とインストール
2. スキルの構造とファイル解説
3. ヘルパー関数の使い方（`weave_helpers` / `wandb_helpers`）
4. Claude Code での活用パターン


In [ ]:
# ローカル環境: uv sync --all-extras で一括インストール可能
# Colab: 以下のセルで個別インストール
!pip install -q uv
!uv pip install --system -q \
  "weave>=0.51.0" \
  "wandb>=0.19.0" \
  "openai>=1.0.0" \
  "python-dotenv>=1.0.0"

In [ ]:
import os

# Colab / ローカル環境の自動判定
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
    print("Google Colab 環境")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv; load_dotenv()
    # ローカル: .env に空欄で書かれていた場合も削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)
    print("ローカル環境")

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")

## 1. wandb/skills とは

```
wandb/skills/
└── skills/
    └── wandb-primary/
        ├── SKILL.md                ← エージェントへの指示書（使い分け・注意事項）
        ├── references/
        │   ├── WEAVE_SDK.md        ← Weave API リファレンス（gotchas 付き）
        │   └── WANDB_SDK.md        ← W&B SDK リファレンス
        └── scripts/
            ├── weave_helpers.py    ← Weave 分析ヘルパー
            └── wandb_helpers.py    ← W&B 実験管理ヘルパー
```

**ベンチマーク結果（Claude Code vs Codex）**

| カテゴリ | Claude Code | Codex |
|---|---|---|
| Weave 分析 | **97%** | 79% |
| Weave ツール操作 | **95%** | 86% |
| モデル学習 | **90%** | 72% |
| 障害検出 | **86%** | 63% |

スキルをインストールすることで、エージェントはこれらのタスクを高精度で実行できます。


## 2. インストール

In [ ]:
# npx でインストール（プロジェクトルートで実行）
# Claude Code 用に追加する場合:
# npx skills add wandb/skills

# インストール後のファイル確認
import os
skills_path = "skills/wandb-primary"
if os.path.exists(skills_path):
    for root, dirs, files in os.walk(skills_path):
        level = root.replace(skills_path, "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in files:
            print(f"{indent}  {f}")
else:
    print("skills/ ディレクトリが見つかりません")
    print("プロジェクトルートで以下を実行してください:")
    print("  npx skills add wandb/skills")


## 3. SKILL.md — エージェントへの指示書

`SKILL.md` はエージェントが W&B / Weave を扱う際の「行動指針」です。

### タスク別 SDK の使い分け

| タスク | 使用 SDK |
|---|---|
| 学習 run・損失曲線・ハイパーパラメータの照会 | W&B SDK (`wandb.Api()`) |
| LLM トレース・eval 結果の照会 | Weave SDK (`weave.init()`, `client.get_calls()`) |
| Weave のラッパー型を Python に変換 | `weave_helpers.unwrap()` |
| 学習 run を DataFrame に変換 | `wandb_helpers.runs_to_dataframe()` |
| eval 結果を分析用リストに展開 | `weave_helpers.eval_results_to_dicts()` |

### エージェントへの重要ルール（SKILL.md より）

1. **トレース・run をデータとして扱う** — 生データをコンテキストに流し込まず、pandas/numpy で統計を計算して届ける
2. **必ず最終回答を出す** — タスクはテーブル + 主要な発見で締める
3. **未知の Weave データには `unwrap()` を使う** — WeaveDict/WeaveObject を検査する前に変換する


## 4. `weave_helpers.py` — Weave 分析ヘルパー

ヘルパー関数は `skills/wandb-primary/scripts/` に配置されています。


In [ ]:
# ヘルパーのインポートパス
import sys
sys.path.insert(0, "skills/wandb-primary/scripts")

# 利用可能な場合はインポート
try:
    from weave_helpers import (
        unwrap,               # WeaveDict/WeaveObject → plain Python
        get_token_usage,      # call からトークン数を抽出
        eval_results_to_dicts,# predict_and_score calls → list of dicts
        pivot_solve_rate,     # タスク別 pivot table
        results_summary,      # eval 結果のコンパクトな要約
        eval_health,          # Evaluation.evaluate call の健全性チェック
        eval_efficiency,      # トークン効率（tokens per success）
    )
    print("weave_helpers インポート成功")
except ImportError:
    print("skills がインストールされていません: npx skills add wandb/skills")


In [ ]:
# unwrap の使用例
# WeaveDict や WeaveObject を plain Python に変換する

import weave
client = weave.init(f"{ENTITY}/{PROJECT}")

# トレースを取得して unwrap
calls = list(client.get_calls(limit=5))
if calls:
    call = calls[0]
    try:
        plain_output = unwrap(call.output)
        print("unwrap 後の output:", plain_output)
        print("型:", type(plain_output))
    except NameError:
        # skills 未インストール時の代替実装例
        def simple_unwrap(obj):
            if hasattr(obj, "keys"):
                return {k: simple_unwrap(obj[k]) for k in obj.keys()}
            elif hasattr(obj, "_val"):
                return simple_unwrap(obj._val)
            return obj
        print("unwrap（代替）:", simple_unwrap(call.output))
else:
    print("トレースがまだありません。他のノートブックを先に実行してください。")


In [ ]:
# get_token_usage の使用例
# LLM 呼び出しが含まれる call からトークン数を抽出

from weave.trace.weave_client import CallsFilter

# LLM 呼び出しを含む call を検索
llm_calls = list(client.get_calls(
    filter=CallsFilter(trace_roots_only=False),
    limit=20,
))

for call in llm_calls[:5]:
    try:
        usage = get_token_usage(call)
        if usage["total_tokens"] > 0:
            print(f"op: {call.op_name.split('/')[-1]}")
            print(f"  input:  {usage['input_tokens']:,} tokens")
            print(f"  output: {usage['output_tokens']:,} tokens")
            print(f"  total:  {usage['total_tokens']:,} tokens")
    except Exception:
        pass


In [ ]:
# eval_results_to_dicts / results_summary の使用例
# Evaluation の predict_and_score calls を分析用 list に展開

from weave.trace.weave_client import CallsFilter

# Evaluation.evaluate の呼び出しを検索
eval_op_ref = f"weave:///{ENTITY}/{PROJECT}/op/Evaluation.evaluate:*"
eval_calls = list(client.get_calls(
    filter=CallsFilter(op_names=[eval_op_ref]),
    limit=5,
))

if eval_calls:
    print(f"Evaluation runs 検出: {len(eval_calls)} 件")
    for ec in eval_calls[:2]:
        print(f"  - {ec.op_name.split('/')[-1]} | started: {ec.started_at}")

    # predict_and_score の子 call を取得して分析
    pas_op_ref = f"weave:///{ENTITY}/{PROJECT}/op/Evaluation.predict_and_score:*"
    pas_calls = list(client.get_calls(
        filter=CallsFilter(
            op_names=[pas_op_ref],
            parent_ids=[eval_calls[0].id],
        ),
        limit=100,
    ))
    print(f"\npredict_and_score calls: {len(pas_calls)} 件")

    try:
        results = eval_results_to_dicts(pas_calls, agent_name="my_model")
        print("\neval_results_to_dicts 結果 (最初の3件):")
        for r in results[:3]:
            print(f"  {r}")
        print("\n", results_summary(results))
    except NameError:
        print("skills 未インストールのため eval_results_to_dicts はスキップ")
else:
    print("Evaluation がまだ実行されていません。03_evaluations.ipynb を先に実行してください。")


## 5. `wandb_helpers.py` — W&B 実験管理ヘルパー


In [ ]:
try:
    from wandb_helpers import (
        runs_to_dataframe,  # runs → pandas DataFrame
        diagnose_run,       # run の収束・過学習を診断
        compare_configs,    # 2 つの run の設定を差分比較
    )
    print("wandb_helpers インポート成功")
except ImportError:
    print("skills がインストールされていません")


In [ ]:
import wandb
import pandas as pd

api = wandb.Api()
path = f"{ENTITY}/{PROJECT}"

# runs を DataFrame に変換
try:
    runs = api.runs(path, order="-created_at")
    if runs:
        try:
            df = pd.DataFrame(runs_to_dataframe(runs, limit=10))
        except NameError:
            # skills 未インストール時の代替
            df = pd.DataFrame([{
                "name": r.name,
                "state": r.state,
                "created_at": r.created_at,
            } for r in list(runs)[:10]])
        print(f"取得した runs: {len(df)} 件")
        print(df.head())
    else:
        print("runs がまだありません")
except Exception as e:
    print(f"runs 取得エラー: {e}")


In [ ]:
# diagnose_run の使用例
# run の収束・過学習を自動診断

try:
    runs_list = list(api.runs(path, order="-created_at", filters={"state": "finished"}))[:5]
    if runs_list:
        run = runs_list[0]
        try:
            diagnosis = diagnose_run(run)
            print(f"Run: {run.name}")
            for k, v in diagnosis.items():
                print(f"  {k}: {v}")
        except NameError:
            print(f"Run: {run.name}")
            print(f"  state: {run.state}")
            print("  （diagnose_run は skills インストール後に利用可能）")
    else:
        print("完了済み runs がありません")
except Exception as e:
    print(f"エラー: {e}")


## 6. Claude Code での活用パターン

`wandb/skills` をインストールした Claude Code は、以下のような自然言語の指示を
正確に解釈してコードを生成・実行できます。

### 実用例

**Weave トレースの分析:**
```
「直近 100 件のトレースのうち、エラーになったものの割合と
 平均トークン数を教えてください」
```

**Evaluation 結果の比較:**
```
「grammar_eval_v1 と grammar_eval_v2 の評価結果を比較して、
 改善したスコアと悪化したスコアをまとめてください」
```

**W&B 実験の診断:**
```
「先週実行した学習 run の中で過学習の兆候があるものを
 特定してください」
```

### ポイント：なぜスキルが必要か

```python
# ❌ スキルなしのエージェントがよくやるミス
weave.init(project="my_project")          # TypeError: positional arg required
client.get_calls(filter={"parent_id": x}) # parent_ids は複数形・リスト
call.output.get("result")                 # WeaveDict は .get() できない場合がある
summary["status"]                         # 正しくは summary["weave"]["status"]

# ✅ スキルありのエージェントが生成するコード
weave.init("entity/project")
client.get_calls(filter=CallsFilter(parent_ids=[x]))
unwrap(call.output).get("result")
summary["weave"]["status"]
```

スキルの `WEAVE_SDK.md` には、このような **gotchas（落とし穴）表** が網羅されており、
エージェントがコンテキストとして参照することで精度が大幅に向上します。


## まとめ

次: **appendix/B_mcp_server.ipynb** → W&B MCP サーバー